Experiment Runner


In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

1. Load configuration

In [ ]:
from src.utils.config import load_configs, apply_overrides
configs = load_configs()

configs = apply_overrides(configs, {
    'experiment': {
        'seeds': [11, 22],
        'budget': {'evaluations': 5},
        'training': {'epochs': 1},
        'data': {'max_train_samples': 4000, 'max_val_samples': 2000, 'max_test_samples': 2000},
    }
})
# --------------------------------------------------------------------------------------------------------------------------

print('algorithm:', configs['experiment']['algorithm'])
print('seeds    :', configs['experiment']['seeds'])
print('budget   :', configs['experiment']['budget'])
print('epochs   :', configs['experiment']['training']['epochs'])

1b. Load the fixed splits

In [ ]:
from src.data import split_loader
train_idx, val_idx, test_idx = split_loader.load_splits()
print('split sizes -> train:', len(train_idx), 'val:', len(val_idx), 'test:', len(test_idx))

2. Run the experiment


In [ ]:
from src.experiment.experiment_manager import ExperimentManager
manager = ExperimentManager(configs=configs, device='auto')
result = manager.run()
print('\nExperiment id:', result['experiment_id'])

In [ ]:
import pandas as pd
from src.tracking import database
runs = pd.DataFrame(database.fetch_all('runs'))
runs[['algorithm', 'seed', 'best_accuracy', 'best_precision', 'best_recall',
      'best_macro_f1', 'total_search_time', 'time_to_best', 'peak_memory_mb']]

In [ ]:
evals = pd.DataFrame(database.fetch_all('evaluations'))
evals[['seed', 'evaluation_number', 'accuracy', 'macro_f1',
       'training_time', 'peak_memory_mb', 'failed']].head(20)

In [ ]:
best = pd.DataFrame(database.fetch_all('best_models'))
best[['run_id', 'evaluation_number', 'accuracy', 'macro_f1', 'model_path']]

In [ ]:
from src.utils import paths
print(paths.RUNS_CSV)
display(pd.read_csv(paths.RUNS_CSV))
print(paths.EVALUATIONS_CSV)
display(pd.read_csv(paths.EVALUATIONS_CSV).head())